# MLP with Node Perturbation

This notebook builds, trains, and visualizes a small multilayer perceptron trained with **node perturbation** -- a biologically plausible alternative to backpropagation -- at hyperparameters already tuned in a companion notebook.

It's intentionally minimal: just the functions and data needed to train the final model and plot its learning curve. The hyperparameter search and the comparison against backprop and Hebbian learning are already done -- they're summarized as text and tables below, not re-run here.

## What is node perturbation?

Node perturbation estimates gradients by random search rather than backpropagation: perturb neuron activity, see whether the loss got better or worse, and reinforce whatever perturbation helped. It's the same family as SPSA and REINFORCE, and formally equivalent to a reward-modulated, three-factor Hebbian plasticity rule.

Unlike simple Hebbian learning, node perturbation needs a single *global* number -- whether the whole network's loss improved -- which doesn't exist until the full forward pass is compared to the label. That means it can't be written as a per-layer custom autograd `Function`. Instead, the implementation below runs a clean and a perturbed forward pass, computes the weight update by hand, and sets `.grad` directly -- the optimizer doesn't care how `.grad` got set, so a standard SGD-style optimizer works unmodified.

One deliberate fix over the course tutorial's reference implementation: output-layer noise is injected into the pre-softmax logits, not the post-softmax probabilities, so the perturbed output always stays a valid probability distribution.

## Setup

In [ ]:
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt
import torch
import torchvision
import contextlib
import io

In [ ]:
import random

def set_seed(seed):
  """Seeds torch, numpy, and Python's random module together, so a given
  seed fixes both weight initialization and dataloader shuffle order."""
  torch.manual_seed(seed)
  np.random.seed(seed)
  random.seed(seed)

set_seed(42)

## Data: MNIST

In [ ]:
def download_mnist(train_prop=0.8, keep_prop=0.5):
  """Downloads MNIST and returns train/valid/test splits. Only `keep_prop`
  of the full dataset is retained, split `train_prop`/`(1 - train_prop)`
  into train/valid."""

  valid_prop = 1 - train_prop
  discard_prop = 1 - keep_prop

  transform = torchvision.transforms.Compose(
      [torchvision.transforms.ToTensor(),
      torchvision.transforms.Normalize((0.1307,), (0.3081,))]
      )

  with contextlib.redirect_stdout(io.StringIO()):  # suppress download output
      full_train_set = torchvision.datasets.MNIST(
          root="./data/", train=True, download=True, transform=transform
          )
      full_test_set = torchvision.datasets.MNIST(
          root="./data/", train=False, download=True, transform=transform
          )

  train_set, valid_set, _ = torch.utils.data.random_split(
      full_train_set,
      [train_prop * keep_prop, valid_prop * keep_prop, discard_prop]
      )
  test_set, _ = torch.utils.data.random_split(
      full_test_set,
      [keep_prop, discard_prop]
      )

  print("Number of examples retained:")
  print(f"  {len(train_set)} (training)")
  print(f"  {len(valid_set)} (validation)")
  print(f"  {len(test_set)} (test)")

  return train_set, valid_set, test_set

In [ ]:
train_set, valid_set, test_set = download_mnist()

## Base model: `MultiLayerPerceptron` and `BasicOptimizer`

`NodePerturbationMultiLayerPerceptron` (below) subclasses this directly -- same forward pass, same weight-initialization bookkeeping (`init_lin1_weight`/`init_lin2_weight`, used later for tracking how far training moves the weights). `BasicOptimizer` is a minimal SGD-style optimizer: for every parameter with a `.grad`, it applies `p.data.add_(p.grad, alpha=-lr)`. It has no idea *how* `.grad` got set -- which is exactly what lets node perturbation's hand-computed updates use it unmodified.

In [ ]:
NUM_INPUTS = np.prod(train_set.dataset.data[0].shape)  # size of an MNIST image
NUM_OUTPUTS = 10  # number of MNIST classes

class MultiLayerPerceptron(torch.nn.Module):
  """Simple multilayer perceptron model class with one hidden layer."""

  def __init__(self, num_inputs=NUM_INPUTS, num_hidden=100, num_outputs=NUM_OUTPUTS,
               activation_type="sigmoid", bias=False):
    super().__init__()

    self.num_inputs = num_inputs
    self.num_hidden = num_hidden
    self.num_outputs = num_outputs
    self.activation_type = activation_type
    self.bias = bias

    self.lin1 = torch.nn.Linear(num_inputs, num_hidden, bias=bias)
    self.lin2 = torch.nn.Linear(num_hidden, num_outputs, bias=bias)

    self._store_initial_weights_biases()
    self._set_activation()
    self.softmax = torch.nn.Softmax(dim=1)

  def _store_initial_weights_biases(self):
    self.init_lin1_weight = self.lin1.weight.data.clone()
    self.init_lin2_weight = self.lin2.weight.data.clone()
    if self.bias:
      self.init_lin1_bias = self.lin1.bias.data.clone()
      self.init_lin2_bias = self.lin2.bias.data.clone()

  def _set_activation(self):
    if self.activation_type.lower() == "sigmoid":
      self.activation = torch.nn.Sigmoid()
    elif self.activation_type.lower() == "tanh":
      self.activation = torch.nn.Tanh()
    elif self.activation_type.lower() == "relu":
      self.activation = torch.nn.ReLU()
    elif self.activation_type.lower() == "identity":
      self.activation = torch.nn.Identity()
    else:
      raise NotImplementedError(f"{self.activation_type} activation type not recognized.")

  def forward(self, X, y=None):
    h = self.activation(self.lin1(X.reshape(-1, self.num_inputs)))
    y_pred = self.softmax(self.lin2(h))
    return y_pred

  def forward_backprop(self, X):
    """Identical to forward(). Kept separate and never overridden by
    subclasses -- it's the fixed reference point other rules get compared
    against (e.g. cosine similarity to the true backprop gradient)."""
    h = self.activation(self.lin1(X.reshape(-1, self.num_inputs)))
    y_pred = self.softmax(self.lin2(h))
    return y_pred

  def list_parameters(self):
    params_list = list()
    for layer_str in ["lin1", "lin2"]:
      params_list.append(f"{layer_str}_weight")
      if self.bias:
        params_list.append(f"{layer_str}_bias")
    return params_list

  def gather_gradient_dict(self):
    params_list = self.list_parameters()
    gradient_dict = dict()
    for param_name in params_list:
      layer_str, param_str = param_name.split("_")
      layer = getattr(self, layer_str)
      grad = getattr(layer, param_str).grad
      if grad is None:
        raise RuntimeError("No gradient was computed")
      gradient_dict[param_name] = grad.detach().clone().numpy()
    return gradient_dict

In [ ]:
class BasicOptimizer(torch.optim.Optimizer):
  """Simple optimizer class based on the SGD optimizer."""

  def __init__(self, params, lr=0.01, weight_decay=0):
    if lr < 0.0:
      raise ValueError(f"Invalid learning rate: {lr}")
    if weight_decay < 0.0:
      raise ValueError(f"Invalid weight_decay value: {weight_decay}")
    defaults = dict(lr=lr, weight_decay=weight_decay)
    super().__init__(params, defaults)

  def step(self):
    for group in self.param_groups:
      for p in group["params"]:
        if p.grad is not None:
          if group["weight_decay"] != 0:
            p.grad = p.grad.add(p, alpha=group["weight_decay"])
          p.data.add_(p.grad, alpha=-group["lr"])

In [ ]:
# Shared config and dataloaders -- full 10-class MNIST, no restriction.
NUM_HIDDEN = 100
ACTIVATION = "sigmoid"
BIAS = False
BATCH_SIZE = 32

train_loader = torch.utils.data.DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
valid_loader = torch.utils.data.DataLoader(valid_set, batch_size=BATCH_SIZE, shuffle=False)

## Plotting utilities

In [ ]:
def get_plotting_color(dataset="train", model_idx=None):
  if model_idx is not None:
    dataset = None
  if model_idx == 0 or dataset == "train":
    color = "#1F77B4"  # blue
  elif model_idx == 1 or dataset == "valid":
    color = "#FF7F0E"  # orange
  elif model_idx == 2 or dataset == "test":
    color = "#2CA02C"  # green
  else:
    raise NotImplementedError(f"{dataset} dataset not recognized.")
  return color


def plot_results(results_dict, num_classes=10, ax=None):
  """Plots classification losses (solid, left axis) and accuracies (dashed,
  right axis) across epochs, for train (blue) and valid (orange)."""

  if ax is None:
    _, ax = plt.subplots(figsize=(7, 3.5))

  loss_ax = ax
  acc_ax = None
  chance = 100 / num_classes

  plotted = False
  for result_type in ["losses", "accuracies"]:
    for dataset in ["train", "valid"]:
      key = f"avg_{dataset}_{result_type}"
      if key in results_dict.keys():
        if result_type == "losses":
          ylabel, plot_ax, ls = "Loss", loss_ax, None
        else:
          if acc_ax is None:
            acc_ax = ax.twinx()
            acc_ax.spines[["right"]].set_visible(True)
            acc_ax.axhline(chance, ls="dashed", color="k", alpha=0.8)
            acc_ax.set_ylim(-5, 105)
          ylabel, plot_ax, ls = "Accuracy (%)", acc_ax, "dashed"

        data = results_dict[key]
        plot_ax.plot(data, ls=ls, label=dataset, alpha=0.8, color=get_plotting_color(dataset))
        plot_ax.set_ylabel(ylabel)
        plotted = True

  if plotted:
    ax.legend(loc="center left")
    ax.set_xticks(range(len(data)))
    ax.set_xticklabels([f"{int(e)}" for e in range(len(data))])
    ymin, ymax = ax.get_ylim()
    if ymin > 0:
      pad = (ymax - ymin) * 0.05
      ax.set_ylim(-pad, ymax + pad)
  else:
    raise RuntimeError("No data found to plot.")

  ax.set_title("Performance across learning")
  ax.set_xlabel("Epoch")
  return ax


def plot_scores_per_class(results_dict, num_classes=10, ax=None):
  """Plots final-epoch accuracy per digit class, train vs. valid, against
  the chance line -- the diagnostic that catches single-class collapse."""

  if ax is None:
    _, ax = plt.subplots(figsize=(6, 3))

  ax.set_prop_cycle(None)
  for s, dataset in enumerate(["train", "valid"]):
    correct_by_class = results_dict[f"{dataset}_correct_by_class"]
    seen_by_class = results_dict[f"{dataset}_seen_by_class"]
    xs, ys = list(), list()
    for i, total in seen_by_class.items():
      xs.append(i + 0.3 * (s - 0.5))
      ys.append(np.nan if total == 0 else 100 * correct_by_class[i] / total)

    avg_key = f"avg_{dataset}_accuracies"
    if avg_key in results_dict.keys():
      ax.axhline(results_dict[avg_key][-1], ls="dashed", alpha=0.8, color=get_plotting_color(dataset))

    ax.bar(xs, ys, label=dataset, width=0.3, alpha=0.8, color=get_plotting_color(dataset))

  ax.set_xticks(range(num_classes))
  ax.set_xlabel("Class")
  ax.set_ylabel("Accuracy (%)")
  ax.set_title("Class scores")
  ax.set_ylim(-5, 105)
  ax.axhline(100 / num_classes, ls="dashed", color="k", alpha=0.8)
  ax.legend()
  return ax

## Node perturbation implementation

In [ ]:
class NodePerturbationMultiLayerPerceptron(MultiLayerPerceptron):
  """
  Node perturbation multilayer perceptron with one hidden layer.

  Node perturbation cannot be written as a per-layer custom autograd
  Function, because its update needs a single GLOBAL number -- whether the
  whole network's loss got better or worse because of this trial's random
  perturbation -- and that number does not exist until the full forward pass
  is compared to the target. A single layer's isolated backward() call has
  no way to see it.

  So instead of using torch.autograd, `perturbed_forward()` runs a clean and
  a perturbed pass under `torch.no_grad()`, and `node_perturbation_step()`
  (below) computes the weight update by hand and sets `.grad` directly.
  `BasicOptimizer` only ever reads `.grad`, so it works unmodified.

  Arguments:
  - noise_std (float, optional): standard deviation of the perturbation
    noise. The single most important hyperparameter here -- see the tuning
    summary below for how it was chosen.
  - antithetic (bool, optional): if True, also computes the update from the
    mirrored perturbation (-xi) and averages the two (variance reduction,
    ~2x compute/step). Not used for the final model below.
  """

  def __init__(self, noise_std=0.3, antithetic=False, **kwargs):
    self.noise_std = noise_std
    self.antithetic = antithetic
    self._np_cache = None
    super().__init__(**kwargs)

    if self.bias:
      raise NotImplementedError(
          "NodePerturbationMultiLayerPerceptron does not yet compute bias "
          "updates. Extend node_perturbation_step() to add bias support "
          "before enabling bias=True here."
          )

  def perturbed_forward(self, X, generator=None):
    """
    Runs a clean and a perturbed forward pass and caches everything
    node_perturbation_step() needs to compute the weight update.

    Noise is injected into the hidden layer's POST-activation firing rate,
    and into the output layer's PRE-softmax logits rather than post-softmax
    probabilities -- this keeps the perturbed output a valid probability
    distribution (a deliberate fix over the course tutorial's reference
    implementation, which perturbs after the softmax).
    """

    X = X.reshape(-1, self.num_inputs)

    with torch.no_grad():
      h_clean = self.activation(self.lin1(X))
      out_clean = self.softmax(self.lin2(h_clean))

      xi_h = self.noise_std * torch.randn(h_clean.shape, generator=generator)
      h_pert = h_clean + xi_h

      out_logits_pert = self.lin2(h_pert)
      xi_out = self.noise_std * torch.randn(out_logits_pert.shape, generator=generator)
      out_pert = self.softmax(out_logits_pert + xi_out)

      self._np_cache = {"X": X, "h_pert": h_pert, "xi_h": xi_h, "xi_out": xi_out}

      if self.antithetic:
        h_pert_neg = h_clean - xi_h
        out_pert_neg = self.softmax(self.lin2(h_pert_neg) - xi_out)
        self._np_cache["h_pert_neg"] = h_pert_neg
        self._np_cache["out_pert_neg"] = out_pert_neg

    return out_clean, out_pert

In [ ]:
def node_perturbation_step(MLP, X, y, criterion_none, optimizer, generator=None):
  """
  One training step of node perturbation. Computes the weight update by hand
  (see NodePerturbationMultiLayerPerceptron docstring for why) and sets
  `.grad` on lin1.weight / lin2.weight, then calls optimizer.step() -- this
  is what lets the existing BasicOptimizer apply the update unmodified.

  Per-example loss (criterion_none) is used, not the batch mean, so each
  example's own perturbation gets credited with its own outcome; averaging
  happens after, not before.
  """

  out_clean, out_pert = MLP.perturbed_forward(X, generator=generator)

  with torch.no_grad():
    eps = 1e-12  # numerical floor so log(0) never happens on a confident wrong prediction
    loss_clean = criterion_none(torch.log(out_clean.clamp_min(eps)), y)
    loss_pert = criterion_none(torch.log(out_pert.clamp_min(eps)), y)

    # positive delta_loss = perturbation REDUCED the loss = reinforce it
    delta_loss = loss_clean - loss_pert

    cache = MLP._np_cache
    batch_size = X.shape[0]

    scaled_xi_h = delta_loss.unsqueeze(1) * cache["xi_h"] / MLP.noise_std**2
    grad_W1_update = scaled_xi_h.t().mm(cache["X"]) / batch_size

    scaled_xi_out = delta_loss.unsqueeze(1) * cache["xi_out"] / MLP.noise_std**2
    grad_W2_update = scaled_xi_out.t().mm(cache["h_pert"]) / batch_size

    if MLP.antithetic:
      loss_pert_neg = criterion_none(torch.log(cache["out_pert_neg"].clamp_min(eps)), y)
      delta_loss_neg = loss_clean - loss_pert_neg
      scaled_xi_h_neg = -delta_loss_neg.unsqueeze(1) * cache["xi_h"] / MLP.noise_std**2
      grad_W1_update_neg = scaled_xi_h_neg.t().mm(cache["X"]) / batch_size
      scaled_xi_out_neg = -delta_loss_neg.unsqueeze(1) * cache["xi_out"] / MLP.noise_std**2
      grad_W2_update_neg = scaled_xi_out_neg.t().mm(cache["h_pert_neg"]) / batch_size
      grad_W1_update = (grad_W1_update + grad_W1_update_neg) / 2
      grad_W2_update = (grad_W2_update + grad_W2_update_neg) / 2

    # BasicOptimizer SUBTRACTS grad * lr -- we want to ADD the update, so negate
    MLP.lin1.weight.grad = -grad_W1_update
    MLP.lin2.weight.grad = -grad_W2_update

  optimizer.step()
  return loss_clean.mean().item()

In [ ]:
def update_results_by_class_in_place(y, y_pred, result_dict, dataset="train", num_classes=10):
  """Updates a results dictionary in place with per-class correct/seen
  counts for one batch -- what plot_scores_per_class() reads from."""

  y_pred = np.argmax(y_pred, axis=1)
  if len(y) != len(y_pred):
    raise RuntimeError("Number of predictions does not match number of targets.")

  for i in result_dict[f"{dataset}_seen_by_class"].keys():
    idxs = np.where(y == int(i))[0]
    result_dict[f"{dataset}_seen_by_class"][int(i)] += len(idxs)
    num_correct = int(sum(y[idxs] == y_pred[idxs]))
    result_dict[f"{dataset}_correct_by_class"][int(i)] += num_correct


def train_epoch_node_perturbation(MLP, train_loader, valid_loader, optimizer, no_train=False, generator=None):
  """Node-perturbation training loop for one epoch. Same results_dict shape
  as a standard backprop epoch; only the inner training-step call differs
  (no loss.backward())."""

  criterion_none = torch.nn.NLLLoss(reduction="none")
  criterion_mean = torch.nn.NLLLoss()

  epoch_results_dict = dict()
  for dataset in ["train", "valid"]:
    for sub_str in ["correct_by_class", "seen_by_class"]:
      epoch_results_dict[f"{dataset}_{sub_str}"] = {i: 0 for i in range(MLP.num_outputs)}

  MLP.train()
  train_losses, train_acc = list(), list()
  for X, y in train_loader:
    if no_train:
      with torch.no_grad():
        y_pred = MLP(X, y=y)
        loss_val = criterion_mean(torch.log(y_pred.clamp_min(1e-12)), y).item()
    else:
      loss_val = node_perturbation_step(MLP, X, y, criterion_none, optimizer, generator=generator)
      with torch.no_grad():
        y_pred = MLP(X, y=y)

    acc = (torch.argmax(y_pred.detach(), axis=1) == y).sum() / len(y)
    train_losses.append(loss_val * len(y))
    train_acc.append(acc.item() * len(y))
    update_results_by_class_in_place(
        y, y_pred.detach(), epoch_results_dict, dataset="train", num_classes=MLP.num_outputs
        )

  num_items = len(train_loader.dataset)
  epoch_results_dict["avg_train_losses"] = np.sum(train_losses) / num_items
  epoch_results_dict["avg_train_accuracies"] = np.sum(train_acc) / num_items * 100

  MLP.eval()
  valid_losses, valid_acc = list(), list()
  with torch.no_grad():
    for X, y in valid_loader:
      y_pred = MLP(X)
      loss = criterion_mean(torch.log(y_pred.clamp_min(1e-12)), y)
      acc = (torch.argmax(y_pred, axis=1) == y).sum() / len(y)
      valid_losses.append(loss.item() * len(y))
      valid_acc.append(acc.item() * len(y))
      update_results_by_class_in_place(y, y_pred.detach(), epoch_results_dict, dataset="valid")

  num_items = len(valid_loader.dataset)
  epoch_results_dict["avg_valid_losses"] = np.sum(valid_losses) / num_items
  epoch_results_dict["avg_valid_accuracies"] = np.sum(valid_acc) / num_items * 100

  return epoch_results_dict


def train_model_node_perturbation(MLP, train_loader, valid_loader, optimizer, num_epochs=5, seed=None):
  """Node-perturbation training loop across epochs. Same return shape as a
  standard backprop run, so it plugs directly into plot_results() etc."""

  generator = None
  if seed is not None:
    generator = torch.Generator()
    generator.manual_seed(seed)

  results_dict = {
      "avg_train_losses": list(), "avg_valid_losses": list(),
      "avg_train_accuracies": list(), "avg_valid_accuracies": list(),
  }

  for e in tqdm(range(num_epochs)):
    no_train = True if e == 0 else False  # baseline epoch, before any training
    latest_epoch_results_dict = train_epoch_node_perturbation(
        MLP, train_loader, valid_loader, optimizer=optimizer, no_train=no_train, generator=generator
        )
    for key, result in latest_epoch_results_dict.items():
      if key in results_dict.keys() and isinstance(results_dict[key], list):
        results_dict[key].append(latest_epoch_results_dict[key])
      else:
        results_dict[key] = result

  return results_dict

## Hyperparameter tuning (summary)

Node perturbation has two key hyperparameters: `noise_std` (how big a random nudge gets injected each step) and `lr` (how big a step to take once a nudge is known to have helped). Both were tuned in stages in a companion notebook -- summarized here as text, not re-run.

### Stage 1 -- sweep `noise_std` (3-class subset, `lr=1e-3` fixed)

| noise_std | mean accuracy | std |
|---|---|---|
| 0.05 | 95.74% | 0.20 |
| 0.10 | 95.74% | 0.20 |
| 0.30 | 95.69% | 0.16 |
| 0.50 | 95.56% | 0.14 |
| 1.00 | 95.10% | 0.17 |
| 2.00 | 91.92% | 0.42 |

Accuracy was flat across most of this range -- the real failure mode was "too large," not "too small": `noise_std=2.0` was both the worst-performing and least reliable setting. `noise_std=0.3` was chosen: within noise of the top scorers, but safely clear of the instability visible at the high end.

### Stage 2 -- sweep `lr` (3-class subset, `noise_std=0.3` fixed)

| lr | mean accuracy | std | weight movement |
|---|---|---|---|
| 0.0001 | 87.25% | 10.47 | 0.39 |
| 0.0005 | 94.47% | 0.34 | 1.50 |
| 0.001 | 95.69% | 0.16 | 2.48 |
| 0.005 | 97.29% | 0.31 | 7.11 |
| 0.01 | 97.68% | 0.23 | 11.62 |
| 0.05 | 97.82% | 0.16 | 52.68 |

Accuracy climbed with `lr` across the whole tested range, but `lr=0.05` moved weights roughly 7-8x their initial scale -- a real instability risk (Hiratani et al., 2022, documents node perturbation's susceptibility to exactly this kind of weight diffusion). `lr=0.01` was chosen: a 0.14-point accuracy cost for a much safer weight trajectory.

### Stage 3 -- joint grid around both winners (3-class subset)

A 1D sweep of each parameter alone can miss interaction effects. The joint grid confirmed `noise_std=0.15, lr=0.01` as the best combination (97.89%), and a 15-epoch, 5-seed confirmation run gave a clean, reliable result: **98.0% ± 0.2% accuracy**.

### Stage 4 -- retuning for the full 10-class task

The 3-class heuristics did not fully transfer: on 10 classes, the direction of the `noise_std` effect actually reversed depending on `lr`, and higher `lr` no longer helped. An extended grid closed this out properly:

| noise_std | lr | mean accuracy | std | weight movement |
|---|---|---|---|---|
| 0.15 | 0.01 | **82.46%** | 3.03 | 61.6 |
| 0.05 | 0.01 | 82.39% | 5.37 | 60.4 |
| 0.05 | 0.02 | 79.64% | 5.37 | 128.5 |
| 0.15 | 0.02 | 77.12% | 5.46 | 129.9 |
| 0.15 | 0.04 | 73.29% | 5.47 | 370.7 |
| 0.05 | 0.04 | 68.73% | 11.83 | 345.7 |

Accuracy, weight movement, and reliability all degrade together as `lr` increases past 0.01 -- a clean, three-way-confirmed instability signature, not noise. `noise_std=0.15, lr=0.01` was confirmed as the right setting for the full task, not just a reasonable guess.

### Final chosen hyperparameters

| | Value |
|---|---|
| `noise_std` | **0.15** |
| `lr` | **0.01** |
| Task | Full 10-class MNIST |
| Epochs (final training) | 15 |

## Training the final model

One run at the tuned settings, on the full 10-class task. Note this is a *single* seed, shown for its learning curve -- the reported comparison result (below) averages 5 seeds to **86.70% ± 0.73%**, so this particular run may land anywhere in that range.

In [ ]:
FINAL_NOISE_STD = 0.15
FINAL_LR_NP = 0.01
FINAL_NUM_EPOCHS = 15
SEED = 0

set_seed(SEED)  # fixes weight initialization
MLP_NP = NodePerturbationMultiLayerPerceptron(
    noise_std=FINAL_NOISE_STD,
    num_hidden=NUM_HIDDEN,
    num_outputs=NUM_OUTPUTS,
    activation_type=ACTIVATION,
    bias=BIAS,
    )
optimizer_np = BasicOptimizer(MLP_NP.parameters(), lr=FINAL_LR_NP)

NP_results_dict = train_model_node_perturbation(
    MLP_NP, train_loader, valid_loader, optimizer_np,
    num_epochs=FINAL_NUM_EPOCHS, seed=SEED,
    )

## Results

In [ ]:
plot_results(NP_results_dict, num_classes=NUM_OUTPUTS);

In [ ]:
plot_scores_per_class(NP_results_dict, num_classes=NUM_OUTPUTS);

## Comparison against backprop and Hebbian

Node perturbation, backprop, and Hebbian were each run 5 times (different seeds) on the identical full 10-class task, 15 epochs each, in a companion notebook. Results:

| Rule | Mean accuracy | Std | Notes |
|---|---|---|---|
| Backprop (15 epochs) | 92.28% | 0.08 | |
| Backprop (5 epochs, original setting) | 89.30% | 0.30 | Significantly different from the 15-epoch result (p < 0.0001) -- backprop was still improving past epoch 5 on this harder task |
| **Node perturbation** | **86.70%** | **0.73** | |
| Hebbian | 10.95% | 0.000000 | Total collapse -- confirmed via per-class accuracy: always predicts class "1," never learns |

One-way ANOVA across the three rules: F=56979.7, p≈0. Pairwise t-tests (Bonferroni-corrected alpha=0.0167 for 3 comparisons), all significant:
- Node perturbation vs. Hebbian: +75.75 pts (p≈0)
- Node perturbation vs. backprop: -5.58 pts (p≈0)
- Hebbian vs. backprop: -81.33 pts (p≈0)

**Interpretation.** Backprop wins outright. Node perturbation is meaningfully behind but genuinely functional -- it learns all ten classes, generalizes, and is roughly 9x noisier run-to-run than backprop (a direct, expected consequence of estimating gradients from a single random perturbation rather than computing them exactly). Hebbian did not solve the task at these settings: it collapsed to predicting a single class for every input, a qualitatively different failure from "less accurate."

**Note for the write-up -- a caveat that must be stated, not omitted:** this comparison is fair in *procedure* (same task, same seeds, same epoch count for all three rules) but not fair in *tuning effort*. Node perturbation went through four rounds of hyperparameter sweeps (Stages 1-4 above); Hebbian ran at the learning rate split the team had already settled on for an easier 3-class task, never independently retuned for the full 10-class problem; backprop used its original default learning rate, never swept at all. The defensible claim is *"node perturbation, tuned, outperformed Hebbian and approached backprop at their existing settings, on this task, across 5 seeds"* -- not *"node perturbation is the better biologically plausible rule in general."* The second claim would require giving Hebbian the same tuning effort first.

## Conclusion

Node perturbation -- a biologically plausible, gradient-free learning rule -- was tuned across four stages and trained on full 10-class MNIST at `noise_std=0.15, lr=0.01`. It reliably learns all ten digit classes, reaching 86.70% ± 0.73% accuracy across 5 seeds: behind backprop's 92.28% (which needs a biologically implausible symmetric weight-transport mechanism to compute its exact gradient), but functioning where Hebbian learning -- at the team's existing settings -- collapsed entirely.

The credit-assignment distinction driving this result: Hebbian's hidden layer only ever sees its own local pre/post-synaptic activity, with no path back to the actual task loss. Node perturbation's every layer is scaled by the same global loss signal, however noisy that signal's estimate is. On this task, global-but-noisy outperformed local-but-blind.